# 02 — CORE Data Exploration

This notebook documents the raw CORE social lettings microdata before any pipeline work. It covers the schema challenge (two incompatible file formats across years), key column distributions, and the data quality issues that the ingestion step had to handle.

**Source**: UK Data Service, SN 9237 — CORE (Continuous Recording of Letting and Sales in Social Housing), 2007–2022.

**A row represents**: One new social housing letting — the moment a household moves into a social rented property. Each letting has details about the property, the household, income, rent, previous situation, and reason for letting.

In [ ]:
import os, glob
os.environ['JAVA_HOME'] = '/opt/homebrew/opt/openjdk@17'
os.environ['PYSPARK_PYTHON'] = '/Users/user/Documents/repos/.venv/bin/python'
os.environ['PYSPARK_DRIVER_PYTHON'] = '/Users/user/Documents/repos/.venv/bin/python'
os.environ['SPARK_LOCAL_IP'] = '127.0.0.1'

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, avg, min, max, trim, when, round as spark_round

spark = SparkSession.builder \
    .master('local[*]') \
    .appName('explore_core') \
    .config('spark.driver.memory', '4g') \
    .getOrCreate()
spark.sparkContext.setLogLevel('ERROR')

BRONZE = '../data/bronze/core_raw/tab'
all_files = sorted(glob.glob(f'{BRONZE}/*.tab'))
print(f'Total tab files: {len(all_files)}')
for f in all_files:
    print(' ', os.path.basename(f))

## 1. The schema problem — two file formats

This is the most important discovery in exploration. The 42 files do not share a common schema.

In [ ]:
# Count columns in each file to reveal schema groups
print(f'{"File":<45} {"Columns":>8}')
print('-' * 55)
for path in all_files:
    name = os.path.basename(path)
    df = spark.read.csv(path, header=True, inferSchema=False, sep='\t')
    print(f'{name:<45} {len(df.columns):>8}')

The column count ranges from 93 to 213. This means **globbing all files together will cause column misalignment** — Spark would use one file's header for all files, putting data into the wrong columns for every other schema variant.

Solution (implemented in `04_ingest_core.ipynb`): read each file individually in a loop, select columns by name (not position), then union.

## 2. Geography — how the UK is divided and how CORE uses it

### UK geography hierarchy

The UK uses a standard hierarchy of geography codes (ONS codes) from largest to smallest:

| Level | Example | Code prefix | Count in England |
|---|---|---|---|
| Country | England | E92 | 1 |
| Government Office Region (GOR) | London | E12 | 9 |
| Local Authority / Borough | Hackney | E09 (London), E06/E07/E08 elsewhere | 33 in London |
| LSOA (Lower Super Output Area) | ~1,500 households | E01 | ~4,835 in London |
| OA (Output Area) | ~300 households | E00 | smallest census unit |

**The 9 Government Office Regions in England:**

| Code | Region |
|---|---|
| E12000001 | North East |
| E12000002 | North West |
| E12000003 | Yorkshire and The Humber |
| E12000004 | East Midlands |
| E12000005 | West Midlands |
| E12000006 | East of England |
| **E12000007** | **London** ← what we filter on |
| E12000008 | South East |
| E12000009 | South West |

Scotland, Wales and Northern Ireland use W, S, N prefixes and are outside the E12 system.

### How CORE uses geography

CORE anonymises data to **region level only** in the public end-user licence version. This means:
- We can identify that a letting is in London (`GOVREG = E12000007`) ✓
- We **cannot** identify which London borough it is in ✗

Borough-level financial vulnerability therefore comes from a separate source — IMD 2019, which is published at LSOA level and tagged with the parent Local Authority (E09 = London borough).

Additionally, **old CORE files (2007–2018) use a numeric shorthand** — `GOVREG = '7'` instead of `'E12000007'`. Our filter handles both:
```python
.filter((trim(col('GOVREG')) == '7') | (trim(col('GOVREG')) == 'E12000007'))
```

In [ ]:
# Check GOVREG values in old vs new files
old = spark.read.csv(f'{BRONZE}/0708_sr_gn_eul.tab', header=True, inferSchema=False, sep='\t')
new = spark.read.csv(f'{BRONZE}/core_lettings_2018-19_eul.tab', header=True, inferSchema=False, sep='\t')

print('=== GOVREG values in old file (0708) ===')
display(old.groupBy('GOVREG').count().orderBy('GOVREG').toPandas().style.format(thousands=","))

print('=== GOVREG values in new file (2018-19) ===')
display(new.groupBy(trim(col('GOVREG')).alias('GOVREG')).count().orderBy('GOVREG').toPandas().style.format(thousands=","))

In [ ]:
# Confirm: GOVREG=7 in old files = London = E12000007 in new files
print('Old file, GOVREG=7 row count:', old.filter(col('GOVREG') == '7').count())
print('New file, GOVREG=E12000007 row count:', new.filter(trim(col('GOVREG')) == 'E12000007').count())
print('\nNote: CORE geography is region-level only — no borough breakdown available in this dataset.')

## 3. Income and rent bands — what do they look like raw?

Income and rent are not stored as numbers — they are string bands. Understanding the format is needed to build the midpoint UDF.

In [ ]:
london_new = new.filter(trim(col('GOVREG')) == 'E12000007')

print('=== Weekly income bands (sample) ===')
display(london_new.groupBy('WEEKINC_T_Bands').count().orderBy('WEEKINC_T_Bands').limit(20).toPandas().style.format(thousands=","))

print('=== Weekly rent bands (sample) ===')
display(london_new.groupBy('WRENT_Bands').count().orderBy('WRENT_Bands').limit(20).toPandas().style.format(thousands=","))

## 4. Blank string issue in integer columns

Several numeric columns contain blank strings instead of nulls. A direct `.cast('int')` will fail — this must be handled before casting.

In [ ]:
int_cols = ['HHMEMBT', 'BEDST', 'BED_MINUS_BEDSTANDARD', 'YEAR']
london_new_cached = london_new.cache()
total = london_new_cached.count()

print(f'London rows in 2018-19 file: {total:,}\n')
print(f'{"Column":<30} {"Blank strings":>15} {"Blank %":>8}')
print('-' * 55)
for c in int_cols:
    if c in london_new.columns:
        blanks = london_new_cached.filter(trim(col(c)) == '').count()
        print(f'{c:<30} {blanks:>15,} {blanks/total*100:>7.1f}%')

## 5. London rows across all years — using the correct filter

In [ ]:
# Read each file individually and count London rows
print(f'{"File":<45} {"London rows":>12}')
print('-' * 59)
total_london = 0
for path in all_files:
    name = os.path.basename(path)
    df = spark.read.csv(path, header=True, inferSchema=False, sep='\t')
    n = df.filter(
        (trim(col('GOVREG')) == '7') | (trim(col('GOVREG')) == 'E12000007')
    ).count()
    total_london += n
    print(f'{name:<45} {n:>12,}')
print('-' * 59)
print(f'{"TOTAL":<45} {total_london:>12,}')

## 6. Key column availability across schema versions

Not all columns exist in all years. These are the columns we want and which years they appear in.

In [ ]:
target_cols = [
    'YEAR', 'GOVREG', 'LETTYPE', 'TENANCY', 'HHMEMBT', 'BEDST',
    'BED_MINUS_BEDSTANDARD', 'BED_MINUS_BEDSTANDARD2',
    'PREVTEN_R', 'REASON_R', 'WEEKINC_T_Bands', 'WRENT_Bands',
    'WTSHORTFALLHB_Bands', 'WTSHORTFALL_Bands',
    'ETHNIC_Bands', 'TENANCYLENGTH_Bands', 'econstat_imputed_R'
]

# Check a sample of files across years
sample_files = [
    '0708_sr_gn_eul.tab', '0910_sr_gn_eul.tab', '1213_sr_gn_eul.tab',
    '1415_sr_gn_eul.tab', '1718_sr_gn_eul.tab', 'core_lettings_2018-19_eul.tab'
]

print(f'{"Column":<30}', end='')
for f in sample_files:
    print(f'{f[:10]:>12}', end='')
print()
print('-' * (30 + 12 * len(sample_files)))

for tc in target_cols:
    print(f'{tc:<30}', end='')
    for f in sample_files:
        df = spark.read.csv(f'{BRONZE}/{f}', header=True, inferSchema=False, sep='\t')
        present = '✓' if tc in df.columns else '✗'
        print(f'{present:>12}', end='')
    print()

## Summary of findings

Key observations that shaped `04_ingest_core.ipynb`:

- **42 files, 7 distinct schemas** (93–213 columns). Cannot glob — must read individually.
- **London filter**: `GOVREG == '7'` for files pre-2018, `GOVREG == 'E12000007'` for 2018–2022.
- **No borough breakdown** — CORE anonymises geography to region level only. Borough-level deprivation must come from a separate source (IMD).
- **Income/rent as string bands** — requires a midpoint UDF for any numeric analysis.
- **Blank strings in integer columns** — `HHMEMBT`, `BEDST`, `BED_MINUS_BEDSTANDARD` all need blank-to-null handling before casting.
- **`econstat_imputed_R`**, **`TENANCYLENGTH_Bands`**, and **`BED_MINUS_BEDSTANDARD`** only appear from 2012–13 onwards — older years get null for these columns.
- **`WTSHORTFALLHB_Bands`** (rent shortfall) is named `WTSHORTFALL_Bands` in some years.